## This notebook can be used to rank a list of nodes from a category that connect to an entity such as a gene. 

In [1]:
from TCT import node_normalizer
from TCT import name_resolver
from TCT import translator_metakg
from TCT import translator_kpinfo
from TCT import translator_query
from TCT import TCT


### Load Translator resources


In [ ]:
from TCT.translator_resources import TranslatorResources
resources = TranslatorResources.load()

## Find the neighborhood of an entity from a subset of APIs 


In [ ]:
# select a list of APIs to use and a list of predicates to use
selected_APIlist = []

filtered = resources.filter(api_list=selected_APIlist) if selected_APIlist else resources
print(filtered.api_names)
print(filtered.meta_kg.shape)

In order to use the neighborhood finder, we have to look up a CURIE ID for a given term.

In [ ]:
name_resolver.lookup(query='BCL2')
name_resolver.lookup(query='ABCB1', return_top_response=False, biolink_type='biolink:Gene', limit=100, only_taxa='NCBITaxon:9606')  # sometimes the identifiers are not in the top 1, users need to check the other returned results

nb_result = TCT.Neiborhood_finder(input_node='NCBIGene:596',
                                   node2_categories=['biolink:Gene', 'biolink:Protein'],
                                   resources=filtered)
input_node_id = nb_result.input_node_id
result = nb_result.knowledge_graph
result_parsed = nb_result.parsed
result_ranked_by_primary_infores = nb_result.ranked

In [ ]:
nb_result = TCT.Neiborhood_finder(input_node='NCBIGene:596',
                                   node2_categories=['biolink:Gene', 'biolink:Protein'],
                                   resources=filtered)
input_node_id = nb_result.input_node_id
result = nb_result.knowledge_graph
result_parsed = nb_result.parsed
result_ranked_by_primary_infores = nb_result.ranked

In [7]:
result

{'00c70ac5-8710-5468-a3cb-dec42260aac3': {'attributes': [{'attribute_source': 'infores:catrax-pharmacogenomics',
    'attribute_type_id': 'biolink:knowledge_level',
    'value': 'knowledge_assertion'},
   {'attribute_source': 'infores:catrax-pharmacogenomics',
    'attribute_type_id': 'biolink:agent_type',
    'value': 'automated_agent'},
   {'attribute_type_id': 'object_category', 'value': 'biolink:Gene'},
   {'attribute_type_id': 'object_name', 'value': 'BCL2'},
   {'attribute_source': 'infores:catrax-pharmacogenomics',
    'attribute_type_id': 'biolink:publications',
    'value': 'PMID:36243968|PMID:10669763',
    'value_type_id': 'biolink:Uriorcurie'},
   {'attribute_type_id': 'knowledge_source', 'value': 'SIGNOR-74939'},
   {'attribute_type_id': 'subject_name', 'value': 'MAPK3'},
   {'attribute_type_id': 'provided_by', 'value': 'SIGNOR'},
   {'attribute_type_id': 'subject_category', 'value': 'biolink:Gene'}],
  'object': 'NCBIGene:596',
  'predicate': 'biolink:regulates',
  'quali

In [ ]:
# Step 8: Visualize the results
TCT.visulization_one_hop_ranking(result_ranked_by_primary_infores=result_ranked_by_primary_infores,
                                result_parsed=result_parsed,
                                num_of_nodes=50,
                                input_query=input_node_id,
                                fontsize=5)

In [9]:
result_ranked_by_primary_infores

,output_node,Name,Num_of_primary_infores,type_of_nodes,unique_predicates
24,NCBIGene:581,BAX,8,subject,"[biolink:regulates, biolink:physically_interac..."
20,NCBIGene:10018,BCL2L11,8,subject,"[biolink:regulates, biolink:physically_interac..."
23,NCBIGene:7157,TP53,8,subject,"[biolink:regulates, biolink:physically_interac..."
10,NCBIGene:5599,MAPK8,8,subject,"[biolink:regulates, biolink:physically_interac..."
8,NCBIGene:5594,MAPK1,7,subject,"[biolink:regulates, biolink:physically_interac..."
...,...,...,...,...,...
974,NCBIGene:3456,IFNB1,1,subject,[biolink:affects]
973,NCBIGene:811,CALR,1,subject,[biolink:affects]
972,NCBIGene:8289,ARID1A,1,subject,[biolink:affects]
971,NCBIGene:3964,LGALS8,1,subject,[biolink:affects]


In [ ]:
from TCT import visualization

dic_graph = visualization.visualize_neighborhood_graph(result=result, show_label=True, height="500", width="100%")

In [11]:
import networkx as nx
from pyvis.network import Network
#G1 = dic_graph['treats']
G2 = dic_graph['directly_physically_interacts_with']
#G3 = dic_graph['_clinical_trials_for']
G_merged = nx.compose_all([G2])
# remove self-loops
G_merged.remove_edges_from(nx.selfloop_edges(G_merged))
# remove duplicate edges
G_merged = nx.Graph(G_merged)
# Select nodes with degree > 10
selected_nodes = ['BCL2']
# select the first neighbors of the selected nodes
for node in list(selected_nodes):
    neighbors = list(G_merged.neighbors(node))
    selected_nodes.extend(neighbors) # expand the selected nodes by adding their neighbors and only keep top 50 nodes
selected_nodes = list(set(selected_nodes))[:100] # keep only top 50 nodes
subgraph = G_merged.subgraph(selected_nodes).copy()

net = Network(height="1000px", width="100%", notebook=True, cdn_resources="in_line")
net.from_nx(subgraph)

# Remove edge labels before passing to PyVis
for u, v, d in subgraph.edges(data=True):
    d.pop("label", None)  # remove 'label' if it exists


for e in net.edges:
    e["title"] = "\n".join([f"{k}: {v}" for k,v in subgraph[e["from"]][e["to"]].items()])
# add title in the figure
title_html = f"<h3>merged_graph</h3>"
net.title = title_html + f"<p>Nodes: {net.num_nodes()} Edges: {net.num_edges()}</p>"
net.show("merged_graph.html")

merged_graph.html


In [12]:
# put G2 into table format
import pandas as pd
edges_data = []
for u, v, d in G2.edges(data=True):
    edge_info = {
        'source': u,
        'target': v,
    }
    edge_info.update(d)  # add all edge attributes
    edges_data.append(edge_info)

edges_df = pd.DataFrame(edges_data)
edges_df.to_csv('BCL2_directly_physically_interacts_with_edges.csv', index=False)

In [13]:
# End of the example
